# Secondary exploratory hybrid evaluation

This notebook compares the original frozen individual models with frozen 50/50 calibrated hybrids on the same 125-person cohort. The cohort was previously evaluated. This is not a fresh independent test, external validation, or confirmatory evidence. The notebook reads an aggregate report only and performs no training or evaluation.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import Markdown, display

def repository_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise RuntimeError('Repository root not found')

report_path = repository_root() / 'reports' / 'generated' / 'secondary-exploratory-hybrid-evaluation-report.json'
report_bytes = report_path.read_bytes()
report = json.loads(report_bytes.decode('utf-8'))
assert report['scope'] == 'secondary_exploratory_evaluation_on_previously_evaluated_test_cohort'
assert report['cohort_previously_evaluated'] is True
assert report['fresh_independent_test'] is False
assert report['external_validation'] is False
assert report['models_refit_or_adjusted_on_test'] is False
assert report['weights_adjusted_on_test'] is False
assert report['thresholds_adjusted_on_test'] is False
assert report['privacy']['participant_identifiers_in_report'] is False
assert report['privacy']['row_level_predictions_in_report'] is False
pd.DataFrame([{'Report': str(report_path.relative_to(repository_root())), 'SHA-256': hashlib.sha256(report_bytes).hexdigest().upper(), 'Participants': report['test_participants'], 'Previously evaluated cohort': report['cohort_previously_evaluated']}])

,Report,SHA-256,Participants,Previously evaluated cohort
0,reports\generated\secondary-exploratory-hybrid...,B0260217A65EC17965173AA1BBBE3726BE2F7195087B70...,125,True


## Frozen configurations and thresholds

In [2]:
domain_labels = {'target_communication_concern': 'Communication', 'target_cognitive_concern': 'Cognitive', 'target_social_emotional_concern': 'Social-emotional'}
configuration_rows = []
for key, domain in report['domains'].items():
    configuration_rows.append({'Domain': domain_labels[key], 'Original model': domain['original_individual']['model_configuration_version'], 'Original threshold': domain['original_individual']['threshold'], 'Hybrid': domain['equal_weight_hybrid']['hybrid_configuration_version'], 'Logistic weight': domain['equal_weight_hybrid']['weights']['logistic'], 'XGBoost weight': domain['equal_weight_hybrid']['weights']['xgboost'], 'Hybrid threshold': domain['equal_weight_hybrid']['threshold']})
pd.DataFrame(configuration_rows).style.format({'Original threshold': '{:.6f}', 'Logistic weight': '{:.2f}', 'XGBoost weight': '{:.2f}', 'Hybrid threshold': '{:.6f}'}).hide(axis='index')

Domain,Original model,Original threshold,Hybrid,Logistic weight,XGBoost weight,Hybrid threshold
Communication,xgboost-regularized-balanced-v1,0.130882,calibrated-soft-voting-equal-weight-v1,0.50,0.50,0.141344
Cognitive,logistic-l2-balanced-v1,0.109680,calibrated-soft-voting-equal-weight-v1,0.50,0.50,0.089746
Social-emotional,xgboost-regularized-balanced-v1,0.126137,calibrated-soft-voting-equal-weight-v1,0.50,0.50,0.167098


## Same-cohort performance comparison

In [3]:
metric_map = {'average_precision_pr_auc': 'PR-AUC', 'roc_auc': 'AUROC', 'accuracy': 'Accuracy', 'balanced_accuracy': 'Balanced accuracy', 'sensitivity': 'Sensitivity', 'specificity': 'Specificity', 'precision': 'PPV', 'negative_predictive_value': 'NPV', 'f1': 'F1', 'brier_score': 'Brier score'}
performance_rows = []
for key, domain in report['domains'].items():
    for result_key, label in (('original_individual', 'Original individual'), ('equal_weight_hybrid', 'Equal-weight hybrid')):
        metrics = domain[result_key]['metrics']; row = {'Domain': domain_labels[key], 'Model': label}
        row.update({label_name: metrics[name] for name, label_name in metric_map.items()}); performance_rows.append(row)
pd.DataFrame(performance_rows).style.format({label: '{:.4f}' for label in metric_map.values()}).hide(axis='index')

Domain,Model,PR-AUC,AUROC,Accuracy,Balanced accuracy,Sensitivity,Specificity,PPV,NPV,F1,Brier score
Communication,Original individual,0.7414,0.9370,0.7440,0.8258,0.9333,0.7182,0.3111,0.9875,0.4667,0.0587
Communication,Equal-weight hybrid,0.7148,0.9430,0.7600,0.8636,1.0000,0.7273,0.3333,1.0000,0.5000,0.0592
Cognitive,Original individual,0.7345,0.9144,0.7680,0.8069,0.8571,0.7568,0.3077,0.9767,0.4528,0.0550
Cognitive,Equal-weight hybrid,0.7630,0.9144,0.7440,0.7934,0.8571,0.7297,0.2857,0.9759,0.4286,0.0560
Social-emotional,Original individual,0.4892,0.8521,0.7760,0.7576,0.7333,0.7818,0.3143,0.9556,0.4400,0.0833
Social-emotional,Equal-weight hybrid,0.5063,0.8679,0.8080,0.7758,0.7333,0.8182,0.3548,0.9574,0.4783,0.0803


## Confusion matrices

In [4]:
confusion_rows = []
for key, domain in report['domains'].items():
    for result_key, label in (('original_individual', 'Original individual'), ('equal_weight_hybrid', 'Equal-weight hybrid')):
        matrix = domain[result_key]['metrics']['confusion_matrix']
        confusion_rows.append({'Domain': domain_labels[key], 'Model': label, 'TN': matrix['true_negative'], 'FP': matrix['false_positive'], 'FN': matrix['false_negative'], 'TP': matrix['true_positive']})
pd.DataFrame(confusion_rows).style.hide(axis='index')

Domain,Model,TN,FP,FN,TP
Communication,Original individual,79,31,1,14
Communication,Equal-weight hybrid,80,30,0,15
Cognitive,Original individual,84,27,2,12
Cognitive,Equal-weight hybrid,81,30,2,12
Social-emotional,Original individual,86,24,4,11
Social-emotional,Equal-weight hybrid,90,20,4,11


## Paired uncertainty

Differences are hybrid minus original. Intervals are descriptive because the cohort was previously evaluated and positive counts are small.

In [5]:
uncertainty_rows = []
for key, domain in report['domains'].items():
    intervals = domain['paired_uncertainty']['hybrid_minus_original_95_percentile_intervals']
    for metric in ('average_precision_pr_auc', 'roc_auc', 'accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'brier_score'):
        point = domain['hybrid_minus_original_metric_differences'][metric]; ci = intervals[metric]
        uncertainty_rows.append({'Domain': domain_labels[key], 'Metric': metric_map[metric], 'Difference': point, '95% interval': f"[{ci['lower_95_percentile']:.4f}, {ci['upper_95_percentile']:.4f}]", 'Interval includes zero': ci['lower_95_percentile'] <= 0 <= ci['upper_95_percentile']})
pd.DataFrame(uncertainty_rows).style.format({'Difference': '{:+.4f}'}).hide(axis='index')

Domain,Metric,Difference,95% interval,Interval includes zero
Communication,PR-AUC,-0.0266,"[-0.1382, 0.0688]",True
Communication,AUROC,+0.0061,"[-0.0158, 0.0315]",True
Communication,Accuracy,+0.0160,"[-0.0320, 0.0720]",True
Communication,Balanced accuracy,+0.0379,"[-0.0182, 0.1182]",True
Communication,Sensitivity,+0.0667,"[0.0000, 0.2000]",True
Communication,Specificity,+0.0091,"[-0.0455, 0.0636]",True
Communication,Brier score,+0.0005,"[-0.0050, 0.0060]",True
Cognitive,PR-AUC,+0.0284,"[-0.0420, 0.1096]",True
Cognitive,AUROC,+0.0000,"[-0.0199, 0.0199]",True
Cognitive,Accuracy,-0.0240,"[-0.0640, 0.0160]",True


## Neutral interpretation

In [6]:
ap_rows = [row for row in uncertainty_rows if row['Metric'] == 'PR-AUC']
uncertain_count = sum(row['Interval includes zero'] for row in ap_rows)
display(Markdown(f"The frozen individual and equal-weight hybrid models were compared on the same previously evaluated cohort. PR-AUC differences had intervals including zero in {uncertain_count} of three domains. Results must be interpreted as secondary exploratory evidence, not independent confirmation of hybrid superiority. No model, calibration, weight, threshold, or deployment setting was changed using these outcomes."))

The frozen individual and equal-weight hybrid models were compared on the same previously evaluated cohort. PR-AUC differences had intervals including zero in 3 of three domains. Results must be interpreted as secondary exploratory evidence, not independent confirmation of hybrid superiority. No model, calibration, weight, threshold, or deployment setting was changed using these outcomes.

## Limitations

The cohort was previously evaluated; the analysis is not external validation; positive counts are small; multiple domains and metrics were inspected; percentile intervals are descriptive; and independent validation remains necessary. The original locked-test report and both earlier notebooks remain the authoritative history of their respective experiments.